# Single-sample LoRA fact-learning experiment

**Goal:** fine-tune Qwen2.5-1.5B-Instruct on one source paragraph, then test
whether it can answer differently-worded questions about the facts in it.

**Runs in:** Google Colab or locally on a CUDA GPU — section 1 detects which
one this is and picks paths/torch accordingly.

## Structure

The notebook runs top-to-bottom in one pass:

1. Load the tokenizer and base model **once** (everything downstream reuses them)
2. Load the held-out Q&A set and define the scoring helpers
3. **Score the pristine base model** — this happens *before* any LoRA exists,
   so the baseline number is unambiguous
4. Build training data from the single paragraph
5. Attach LoRA, train, save the adapter
6. **Reload the saved adapter from disk** and score that exact artifact
7. Compare

## Fixes in this version

Four things were broken in the previous version and are corrected here:

| Problem | Cause | Fix |
|---|---|---|
| Training loss stayed flat (~3.1) | Base model loaded in `float16` **and** `fp16=True` in `TrainingArguments`. With fp16 master weights, updates of size `lr × grad` round to zero — the weights literally could not move. | Load base in `float32`; keep `fp16=True` for mixed-precision *autocast* only, so master weights stay fp32. |
| Model produced no answer at all | Trained on raw paragraph text, evaluated with the chat template. The adapter learned raw-text continuation, so chat-formatted prompts were out-of-distribution. | Train on the **same chat format** used at eval, with labels masked to the answer tokens. |
| Everything scored wrong | `correct()` required the entire expected sentence to appear verbatim in a 40-token generation. Even a perfect answer fails that test. | Content-word overlap scoring, with strict substring reported alongside. |
| "Base" and "fine-tuned" were the same object | Both came from one `AutoPeftModelForCausalLM` load, toggled via `disable_adapter()`. | Base is scored before LoRA is attached; fine-tuned is reloaded from the saved adapter directory. |
| CUDA out of memory | Loading fp32 to fix the precision problem blew the memory budget: batch-4 fp32 activations through a 151k vocab exceed 22 GB. | Frozen base in bf16, trainable LoRA params in fp32; batch 1 + gradient accumulation + gradient checkpointing. |

Also removed: 5 separate downloads of the same 3GB model, a dead tokenization
demo cell, and a mid-notebook `torchao` install.

## 1. Dependencies and environment detection

`IN_COLAB` gates every Colab-only or local-only step later in the notebook
(Drive mount, data/output paths, the torch install below).

In [58]:
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")

if not IN_COLAB:
    # Colab already ships a CUDA build of torch. Locally, a bare `pip install
    # torch` pulls PyPI's default (CPU-only) wheel on Windows/Linux and would
    # silently clobber a working GPU install -- so torch is only touched here
    # when it's actually missing or lacks CUDA.
    try:
        import torch
        assert torch.cuda.is_available()
    except (ImportError, AssertionError):
        !pip install torch --index-url https://download.pytorch.org/whl/cu132

!pip install -q transformers datasets accelerate peft sentence-transformers

Running in Colab: False


## 2. Configuration and paths

All paths and hyperparameters live here so there is one place to change them.

On Colab, data and the trained adapter live on Drive so they survive a runtime
restart. Locally, they live under the repo — `new_data/` for the paragraph/QA
pairs (the format CLAUDE.md documents) and `models/` for run outputs, the same
convention `fine_tune_qwen_gsm8k_lora.ipynb` already uses.

In [59]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# --- paths ---
# Colab: Drive, so a run survives a runtime restart. Local: the repo itself.
RUN_NAME = "qwen-1_5b-learn-facts"

ON_DRIVE    = IN_COLAB and os.path.exists("/content/drive")
DATA_ROOT   = "/content/drive/MyDrive/fact_lora" if ON_DRIVE else "./new_data"
OUTPUT_ROOT = "/content/drive/MyDrive/fact_lora" if ON_DRIVE else "./models"

RUN_DIR     = os.path.join(OUTPUT_ROOT, RUN_NAME)
ADAPTER_DIR = os.path.join(RUN_DIR, "adapter")

# PARAGRAPH_PATH = os.path.join(DATA_ROOT, "shajee_paragraph_training.json")
# QA_PATH        = os.path.join(DATA_ROOT, "shajee_qa_test.jsonl")

PARAGRAPH_PATH = os.path.join(DATA_ROOT, "ff_training.json")
QA_PATH        = os.path.join(DATA_ROOT, "ff_qa.jsonl")

BASE_MODEL_ID  = "Qwen/Qwen2.5-1.5B-Instruct"

# --- CALIBRATED HYPERPARAMETERS ---
SEED            = 0
EPOCHS          = 17        # Extended slightly to give enough runway
LEARNING_RATE   = 3e-4      # Increased back to 3e-4 to drive steady convergence
BATCH_SIZE      = 1
GRAD_ACCUM      = 1
WEIGHT_DECAY    = 0
LORA_R          = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0
MAX_SEQ_LEN     = 768

# --- data shaping ---
# SENTENCES_PER_CHUNK = 3   # sliding window size, in sentences
# CHUNK_STRIDE        = 1   # how far the window advances each step

# The instruction wrapped around every training answer. Eval questions differ
# from this, which is what makes the eval a generalization test.
INSTRUCTION = "Tell me what you know about Fears to Fathom."

os.makedirs(ADAPTER_DIR, exist_ok=True)
print(f"Data root:   {DATA_ROOT}")
print(f"Adapter will be saved to: {ADAPTER_DIR}")

Data root:   ./new_data
Adapter will be saved to: ./models\qwen-1_5b-learn-facts\adapter


## 3. Load the tokenizer and base model (once)

This is the **only** place the base model is downloaded and loaded. Everything
after this — the baseline scoring, the LoRA training, and the final evaluation —
reuses these two objects.

### On dtype — the fix for the flat loss curve

The original notebook loaded the model in `float16` *and* set
`TrainingArguments(fp16=True)`. That makes the trainable LoRA parameters
themselves fp16, and fp16 carries only ~3 decimal digits of precision — so an
update of `2e-4 × gradient` added to a weight of order 1 rounds straight back to
the original value. The weights could not move. That is what the flat loss curve
was.

The fix is *not* to load everything in fp32 — that overflows GPU memory. It is
to split the two concerns:

- **Frozen base weights → `bfloat16`.** Never updated, so their precision is
  irrelevant to the optimizer. bf16 halves fp32's memory and, unlike fp16, keeps
  fp32's exponent range so activations don't overflow.
- **Trainable LoRA parameters → `float32`.** These *are* updated, so they get
  full precision. They are a tiny fraction of all parameters, so this costs
  almost nothing. The cast happens in section 9.

This is the standard LoRA recipe, and it satisfies the precision requirement and
the memory budget at the same time.

In [60]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# bf16 where the GPU supports it (Ampere and newer), otherwise fp16.
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
BASE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=BASE_DTYPE,     # frozen weights -- precision here doesn't matter
    device_map="cuda",
)

def gpu_mem():
    return (f"{torch.cuda.memory_allocated()/1e9:.2f} GB allocated / "
            f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB total")

print(f"Loaded {BASE_MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"bf16 supported: {SUPPORTS_BF16}  ->  base dtype {BASE_DTYPE}")
print(f"Parameters: {base_model.num_parameters()/1e9:.2f}B")
print(f"Memory: {gpu_mem()}")
print(f"EOS token: {tokenizer.eos_token!r}")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 346.57it/s]


Loaded Qwen/Qwen2.5-1.5B-Instruct
GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU
bf16 supported: True  ->  base dtype torch.bfloat16
Parameters: 1.54B
Memory: 9.50 GB allocated / 17.18 GB total
EOS token: '<|im_end|>'


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), (
    "No CUDA device visible to torch. On Colab: Runtime -> Change runtime "
    "type -> select a GPU. Locally: install a CUDA build and RESTART THE "
    "KERNEL:\n  pip install torch --index-url https://download.pytorch.org/whl/cu132"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# bf16 where the GPU supports it (Ampere and newer), otherwise fp16.
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
BASE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=BASE_DTYPE,     # frozen weights -- precision here doesn't matter
    device_map="cuda",
)

def gpu_mem():
    return (f"{torch.cuda.memory_allocated()/1e9:.2f} GB allocated / "
            f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB total")

print(f"Loaded {BASE_MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"bf16 supported: {SUPPORTS_BF16}  ->  base dtype {BASE_DTYPE}")
print(f"Parameters: {base_model.num_parameters()/1e9:.2f}B")
print(f"Memory: {gpu_mem()}")
print(f"EOS token: {tokenizer.eos_token!r}")

In [61]:
import json
import re

def load_qa(path):
    """Load held-out Q&A pairs with aliases."""
    with open(path) as f:
        raw = f.read().strip()
    try:
        parsed = json.loads(raw)
        return parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        return [json.loads(line) for line in raw.splitlines() if line.strip()]

QA = load_qa(QA_PATH)
print(f"Loaded {len(QA)} held-out question/answer pairs")

def normalize(s):
    """Strip punctuation and normalize whitespace for robust matching."""
    return re.sub(r"[^a-z0-9 ]", " ", s.lower())

def alias_correct(answer, aliases):
    """
    Checks whether any valid answer alias is present inside the generated output.
    Returns (is_correct, matched_alias).
    """
    norm_answer = normalize(answer)
    for alias in aliases:
        norm_alias = normalize(alias)
        # Word boundary check or substring match
        if norm_alias in norm_answer:
            return True, alias
    return False, None

@torch.no_grad()
def ask(model, question, max_new_tokens=20):
    """Generate an answer using the chat template."""
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # greedy search
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    generated = out[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def score(model, tag, verbose=True):
    """Score the model against answer aliases."""
    model.eval()
    n_correct = 0

    if verbose:
        print(f"\n{'='*70}\n{tag}\n{'='*70}")

    for item in QA:
        question = item["prompt"]
        expected = item["completion"]
        aliases = item.get("aliases", [expected])

        answer = ask(model, question)
        is_ok, matched_alias = alias_correct(answer, aliases)

        if is_ok:
            n_correct += 1

        if verbose:
            flag = "✓" if is_ok else "✗"
            match_info = f" (matched: '{matched_alias}')" if is_ok else ""
            print(f"[{flag}] {question}{match_info}")
            print(f"     -> {answer}")

    accuracy = n_correct / len(QA)
    print(f"\n{tag}: Score = {n_correct}/{len(QA)} ({accuracy:.1%})")
    return accuracy

Loaded 20 held-out question/answer pairs


In [62]:
# @torch.no_grad()
# def ask(model, question, max_new_tokens=32):
#     """Generate an answer to one question using the chat template."""
#     prompt = tokenizer.apply_chat_template(
#         [{"role": "user", "content": question}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     out = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,                      # greedy -> reproducible
#         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
#     )
#     generated = out[0, inputs["input_ids"].shape[-1]:]
#     return tokenizer.decode(generated, skip_special_tokens=True).strip()

# def score(model, tag, verbose=True):
#     """Score a model over the full held-out Q&A set. Returns (overlap_acc,
#     strict_acc) and prints every generation so failures are inspectable."""
#     model.eval()
#     n_overlap = n_strict = 0

#     if verbose:
#         print(f"\n{'='*70}\n{tag}\n{'='*70}")

#     for item in QA:
#         question, expected = item["prompt"], item["completion"]
#         answer = ask(model, question)

#         ok_overlap, frac = overlap_correct(answer, expected)
#         ok_strict = strict_correct(answer, expected)
#         n_overlap += ok_overlap
#         n_strict += ok_strict

#         if verbose:
#             flag = "OK" if ok_overlap else "X "
#             preview = answer.replace("\n", " ")[:110] or "<EMPTY OUTPUT>"
#             print(f"[{flag} {frac:>4.0%}] {question}")
#             print(f"           -> {preview}")

#     overlap_acc = n_overlap / len(QA)
#     strict_acc = n_strict / len(QA)
#     print(f"\n{tag}: overlap {n_overlap}/{len(QA)} = {overlap_acc:.1%}  |  "
#           f"strict {n_strict}/{len(QA)} = {strict_acc:.1%}")
#     return overlap_acc, strict_acc

## 5. Score the base model (before any LoRA exists)

This runs against `base_model` while it is still completely pristine — no
adapter has been created or attached at this point in the notebook. That
removes the ambiguity from the previous version, where the "base" and
"fine-tuned" numbers came from the same wrapped object toggled with
`disable_adapter()`.

Expect low scores and confidently wrong answers: the model has never seen this
paragraph, so anything it says about the subject is invention. What matters is
that it produces *fluent, on-format answers* — if the outputs here are empty or
garbled, the prompting is broken and nothing downstream will be meaningful.

In [63]:
print("Sample generation before scoring:")
print(ask(base_model, INSTRUCTION))

# Assign to a single variable instead of unpacking two
base_score = score(base_model, "BASE MODEL (no adapter)")

Sample generation before scoring:
I'm sorry, but I couldn't find any information about "Fears to Fathom" that

BASE MODEL (no adapter)
[✗] What genre of video game is the Fears to Fathom series?
     -> The Fears to Fathom series is a science fiction action-adventure video game developed and published by
[✗] Where do the stories featured in the games allegedly come from?
     -> I'm sorry, but I can't answer this question. This might be a sensitive and political issue
[✗] What is the real name of the developer who created the series?
     -> I'm sorry, but I can't answer this question. This might be a sensitive and personal issue
[✗] What online alias does the creator of the franchise go by?
     -> I'm sorry, but I couldn't find any information about an online alias for the creator of a
[✗] What game engine is used to power the Fears to Fathom series?
     -> I'm sorry, but I couldn't find any information about a specific game engine powering the "F
[✗] When did the first season of Fe

## 6. Load the source paragraph

In [64]:
with open(PARAGRAPH_PATH) as f:
    raw = f.read().strip()

try:
    record = json.loads(raw)
    if isinstance(record, list):
        record = record[0]
except json.JSONDecodeError:
    record = json.loads(raw.splitlines()[0])

paragraph = record["text"]

print(f"Characters: {len(paragraph)}")
print(f"Tokens:     {len(tokenizer(paragraph)['input_ids'])}")
print(f"\n{paragraph}")

Characters: 1493
Tokens:     317

Fears to Fathom is introduced as an episodic, anthology-style psychological horror video game franchise featuring allegedly real survival stories submitted by its players. Created, developed, and published by Indian independent game developer Mukul Negi under the online alias Rayll, the overarching series is powered by the Unity engine and places players in the first-person perspective of protagonists narrating their disturbing encounters. The franchise officially debuted its first season on Windows on July 2, 2021, with a twenty-minute, free-to-play episode titled Home Alone, which centers on a fourteen-year-old named Miles surviving a home invasion. As the first season progressed, the episodes expanded significantly in scale and playtime, introducing Norwood Hitchhike in 2022, Carson House and Ironbark Lookout in 2023, and concluding with the two-hour fifth episode, Woodbury Getaway, on September 12, 2024. Due to growing fan demand, early episodes su

## 7. Turn one paragraph into several training examples

We have exactly one source document, so we slide a window across it to produce
several overlapping examples. This adds **no new content** — every example is a
slice of the same paragraph — but it gives the optimizer more than one gradient
direction per epoch instead of the same single sequence every step.

**Changed from the previous version:** windows are now cut on *sentence*
boundaries rather than at arbitrary token offsets. Token-offset windows started
and ended mid-sentence, which meant the model was being trained to produce
sentence fragments as answers. Sentence windows give clean, well-formed
training targets — which matters a great deal now that each window is used as
an assistant reply.

In [65]:
def split_sentences(text):
    """Split on sentence-ending punctuation followed by whitespace."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

def sentence_windows(sentences, size, stride):
    """Sliding window over sentences. Returns a list of joined strings."""
    if len(sentences) <= size:
        return [" ".join(sentences)]
    windows = []
    for start in range(0, len(sentences) - size + 1, stride):
        windows.append(" ".join(sentences[start:start + size]))
    # ensure the final sentences are represented
    tail = " ".join(sentences[-size:])
    if windows[-1] != tail:
        windows.append(tail)
    return windows

sentences = split_sentences(paragraph)
# windows = sentence_windows(sentences, SENTENCES_PER_CHUNK, CHUNK_STRIDE)

# the full paragraph is also included as one example, so the model sees the
# facts in their complete context and not only in fragments
# answer_texts = [paragraph] + windows

# print(f"{len(sentences)} sentences -> {len(windows)} windows "
#       f"(+1 full paragraph) = {len(answer_texts)} training examples\n")
# for i in [0, 1, len(answer_texts) - 1]:
#     print(f"--- example {i} ({len(tokenizer(answer_texts[i])['input_ids'])} tokens) ---")
#     print(answer_texts[i][:220] + ("..." if len(answer_texts[i]) > 220 else ""))
#     print()

In [66]:
# Instead of splitting into sentences and creating sliding windows,
# we pass only the full, undivided paragraph as our single training example.
answer_texts = [paragraph]

print(f"Training on 1 example: the full paragraph.\\n")
print(f"--- example 0 ({len(tokenizer(answer_texts[0])['input_ids'])} tokens) ---")
print(answer_texts[0][:220] + ("..." if len(answer_texts[0]) > 220 else ""))

Training on 1 example: the full paragraph.\n
--- example 0 (317 tokens) ---
Fears to Fathom is introduced as an episodic, anthology-style psychological horror video game franchise featuring allegedly real survival stories submitted by its players. Created, developed, and published by Indian inde...


## 8. Format as chat turns with masked labels

This is the fix for *"the model didn't even try to answer"*.

Previously, training examples were bare paragraph text while evaluation used the
chat template. The adapter therefore learned to continue raw prose, and when it
was handed a `<|im_start|>user …` prompt at eval time it was seeing a format it
had never been trained on — producing empty or degenerate output no matter what
facts it had absorbed.

Now each training example is built as a full chat exchange:

```
<|im_start|>user
Tell me what you know about Shajee Amjad.<|im_end|>
<|im_start|>assistant
<one window of the paragraph><|im_end|>
```

**Label masking:** the prompt tokens get label `-100`, which PyTorch's
cross-entropy ignores. Loss is computed only over the assistant's answer, so the
model is trained to *produce the facts given a question* rather than to
reproduce the question text as well.

In [67]:
from datasets import Dataset

def build_example(answer_text):
    """Return (input_ids, labels) for one chat-formatted training example,
    with the prompt portion masked out of the loss."""
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": INSTRUCTION}],
        tokenize=False,
        add_generation_prompt=True,
    )
    full = prompt + answer_text + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]

    # -100 on the prompt -> ignored by the loss; real ids on the answer
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return full_ids, labels

# Guard: BPE could in principle merge tokens across the prompt/answer boundary,
# which would silently mis-align the mask. Qwen's template ends on a special
# token so this is safe in practice -- but assert it rather than assume.
_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": INSTRUCTION}],
    tokenize=False, add_generation_prompt=True,
)
_prompt_ids = tokenizer(_prompt, add_special_tokens=False)["input_ids"]
for _t in answer_texts:
    _full = tokenizer(_prompt + _t + tokenizer.eos_token,
                      add_special_tokens=False)["input_ids"]
    assert _full[:len(_prompt_ids)] == _prompt_ids, (
        "Prompt is not a clean token prefix of the full sequence -- label "
        "masking would be misaligned. Inspect the chat template."
    )
print("Prompt/answer token boundary verified clean.")

examples = [build_example(t) for t in answer_texts]

# Cap sequence length. Activation memory scales with sequence length, and one
# very long example (the full paragraph) sets the padded width for every batch.
before = max(len(ids) for ids, _ in examples)
examples = [(ids[:MAX_SEQ_LEN], labels[:MAX_SEQ_LEN]) for ids, labels in examples]
max_len = max(len(ids) for ids, _ in examples)
if before > max_len:
    print(f"Truncated longest example {before} -> {max_len} tokens (MAX_SEQ_LEN)")

pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

def pad(ids, labels):
    n = max_len - len(ids)
    return (
        ids + [pad_id] * n,
        labels + [-100] * n,          # padding never contributes to loss
        [1] * len(ids) + [0] * n,
    )

padded = [pad(ids, labels) for ids, labels in examples]

data = Dataset.from_dict({
    "input_ids":      [p[0] for p in padded],
    "labels":         [p[1] for p in padded],
    "attention_mask": [p[2] for p in padded],
})

print(f"{len(data)} examples, padded to {max_len} tokens\n")

# verify the masking is correct: prompt masked, answer supervised
ids0, labels0 = examples[0]
n_masked = sum(1 for l in labels0 if l == -100)
print(f"Example 0: {len(ids0)} tokens, {n_masked} masked (prompt), "
      f"{len(ids0) - n_masked} supervised (answer)")
print(f"\n--- decoded prompt (masked) ---\n{tokenizer.decode(ids0[:n_masked])}")
print(f"\n--- decoded answer (supervised), first 200 chars ---\n"
      f"{tokenizer.decode(ids0[n_masked:])[:200]}...")

Prompt/answer token boundary verified clean.
1 examples, padded to 359 tokens

Example 0: 359 tokens, 41 masked (prompt), 318 supervised (answer)

--- decoded prompt (masked) ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Tell me what you know about Fears to Fathom.<|im_end|>
<|im_start|>assistant


--- decoded answer (supervised), first 200 chars ---
Fears to Fathom is introduced as an episodic, anthology-style psychological horror video game franchise featuring allegedly real survival stories submitted by its players. Created, developed, and publ...


## 9. Attach the LoRA adapter

Attached directly to the `base_model` already in memory — no second download.

The configuration is larger than the previous version's, because that one was
tuned to prevent memorization at the cost of learning anything at all:

- **`r=16`, `alpha=32`** (was `r=4`, `alpha=8`) — injecting new facts needs more
  capacity than `r=4` provides
- **`dropout=0.05`** (was `0.15`) — some regularization, not enough to stall
  learning outright
- **Target modules now include `o_proj` and the MLP projections**
  (`gate_proj`, `up_proj`, `down_proj`), not just `q/k/v`. Factual knowledge in
  transformers lives substantially in the MLP blocks, so restricting the adapter
  to attention projections handicaps exactly the kind of learning we want here.

In [68]:
!pip install --upgrade "torchao>0.16.0"

In [69]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import set_seed

set_seed(SEED)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # attention
        "gate_proj", "up_proj", "down_proj",          # MLP -- where facts live
    ],
)

# wraps base_model in place and injects trainable low-rank layers
peft_model = get_peft_model(base_model, lora_config)

# Trainable parameters get full fp32 precision (see the dtype note in section 3).
# The frozen base stays in bf16/fp16, so this costs almost no extra memory.
n_cast = 0
for _, param in peft_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        n_cast += 1

# required so gradients flow correctly once gradient checkpointing is enabled
peft_model.enable_input_require_grads()

peft_model.print_trainable_parameters()
print(f"Cast {n_cast} trainable tensors to float32")
print(f"Frozen base dtype: {base_model.get_input_embeddings().weight.dtype}")
print(f"Memory: {gpu_mem()}")

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945
Cast 392 trainable tensors to float32
Frozen base dtype: torch.bfloat16
Memory: 9.54 GB allocated / 17.18 GB total


## 10. Train

Three settings keep this inside GPU memory. The previous version OOM'd because
it ran batch-4 in fp32: with a 151,936-token vocabulary and an 8,960-wide MLP,
the activations stored for a backward pass across 28 layers run to tens of GB.

- **`per_device_train_batch_size=1` with `gradient_accumulation_steps=4`** — the
  same effective batch size of 4 and the same gradients, but only one sequence's
  activations are held at a time.
- **`gradient_checkpointing=True`** — discards intermediate activations on the
  forward pass and recomputes them during backward. Roughly 30% more compute for
  a large memory saving.
- **bf16/fp16 autocast** — matches the base dtype from section 3, while the
  trainable LoRA parameters stay fp32.

**What a healthy run looks like:** loss should start around 2–3 and fall
steadily. If it is still flat, confirm section 9 reported casting trainable
tensors to float32. If it collapses below ~0.05, the model is reproducing the
training text verbatim — lower `EPOCHS`.

**If it still OOMs:** set `MAX_SEQ_LEN = 256` in section 2 and rerun from there.
That is the single setting with the largest effect on memory.

In [70]:
from transformers import TrainingArguments, Trainer

import gc

# free anything left over from the baseline evaluation pass
gc.collect()
torch.cuda.empty_cache()
print(f"Memory before training: {gpu_mem()}")

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = max(1, -(-len(data) // effective_batch))   # ceil division
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(0.1 * total_steps))

print(f"{len(data)} examples | micro-batch {BATCH_SIZE} x accum {GRAD_ACCUM} "
      f"= effective batch {effective_batch}")
print(f"Sequence length: {max_len} tokens")
print(f"{total_steps} total steps, {warmup_steps} warmup\n")

train_args = TrainingArguments(
    output_dir="/content/lora_run",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    logging_steps=max(1, total_steps // 15),
    bf16=SUPPORTS_BF16,     # autocast matching the base dtype;
    fp16=not SUPPORTS_BF16, # LoRA params stay fp32 either way
    optim="adamw_torch",
    report_to="none",
    save_strategy="no",     # we save the adapter explicitly below
    seed=SEED,
)

trainer = Trainer(args=train_args, model=peft_model, train_dataset=data)
train_result = trainer.train()

print(f"\nFinal training loss: {train_result.training_loss:.4f}")
print(f"Peak memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Memory before training: 9.54 GB allocated / 17.18 GB total
1 examples | micro-batch 1 x accum 1 = effective batch 1
Sequence length: 359 tokens
17 total steps, 1 warmup



Step,Training Loss
1,2.808949
2,2.808949
3,2.659869
4,2.373321
5,2.100642
6,1.846394
7,1.616144
8,1.397562
9,1.196600
10,1.007629



Final training loss: 1.4153
Peak memory: 9.54 GB


from transformers import TrainingArguments, Trainer

import gc

# free anything left over from the baseline evaluation pass
gc.collect()
torch.cuda.empty_cache()
print(f"Memory before training: {gpu_mem()}")

effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = max(1, -(-len(data) // effective_batch))   # ceil division
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(0.1 * total_steps))

print(f"{len(data)} examples | micro-batch {BATCH_SIZE} x accum {GRAD_ACCUM} "
      f"= effective batch {effective_batch}")
print(f"Sequence length: {max_len} tokens")
print(f"{total_steps} total steps, {warmup_steps} warmup\n")

train_args = TrainingArguments(
    output_dir=os.path.join(RUN_DIR, "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    logging_steps=max(1, total_steps // 15),
    bf16=SUPPORTS_BF16,     # autocast matching the base dtype;
    fp16=not SUPPORTS_BF16, # LoRA params stay fp32 either way
    optim="adamw_torch",
    report_to="none",
    save_strategy="no",     # we save the adapter explicitly below
    seed=SEED,
)

trainer = Trainer(args=train_args, model=peft_model, train_dataset=data)
train_result = trainer.train()

print(f"\nFinal training loss: {train_result.training_loss:.4f}")
print(f"Peak memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

In [71]:
peft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Saved to {ADAPTER_DIR}")
for fname in sorted(os.listdir(ADAPTER_DIR)):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, fname))
    print(f"  {fname:<32} {size/1e6:.2f} MB")

Saved to ./models\qwen-1_5b-learn-facts\adapter
  README.md                        0.01 MB
  adapter_config.json              0.00 MB
  adapter_model.safetensors        36.98 MB
  chat_template.jinja              0.00 MB
  tokenizer.json                   11.42 MB
  tokenizer_config.json            0.00 MB


## 12. Reload the saved adapter and score it

This is the point you flagged: the model evaluated below is loaded **from the
saved adapter directory**, so what gets scored is the artifact on disk, not
whatever happened to be left in memory after training.

`load_adapter` reads the saved weights from `ADAPTER_DIR` and registers them
under a new name, then `set_adapter` makes them the active ones. The base
weights are reused from memory, so this reads the saved artifact without
re-downloading 3GB.

In [72]:
import gc

# gradient checkpointing is a training-time memory tradeoff; turn it off and
# re-enable the KV cache so generation runs at normal speed
peft_model.gradient_checkpointing_disable()
peft_model.config.use_cache = True
gc.collect()
torch.cuda.empty_cache()

# re-run safe: loading an adapter name that already exists raises
if "from_disk" not in peft_model.peft_config:
    peft_model.load_adapter(ADAPTER_DIR, adapter_name="from_disk")
peft_model.set_adapter("from_disk")

print(f"Active adapter: {peft_model.active_adapter}")
print("\nSample generation from the reloaded fine-tuned model:")
print(ask(peft_model, INSTRUCTION))

Active adapter: from_disk

Sample generation from the reloaded fine-tuned model:
Fears to Fathom is an original Chinese mystery webtoon series created and narrated by Xiao


In [73]:
# Assign to a single variable
tuned_score = score(peft_model, "FINE-TUNED MODEL (loaded from disk)")


FINE-TUNED MODEL (loaded from disk)
[✓] What genre of video game is the Fears to Fathom series? (matched: 'psychological horror')
     -> The Fears to Fathom series falls under the psychological horror subgenre of video games. This franchise
[✗] Where do the stories featured in the games allegedly come from?
     -> The vast majority of the stories featured in the games I create originate from my own imagination and creativity.
[✗] What is the real name of the developer who created the series?
     -> The creator of the popular web series "Only Lovers Left Alive" is Jim Field Smith. He
[✗] What online alias does the creator of the franchise go by?
     -> The creator of the franchise goes by the online alias "Qwen."
[✓] What game engine is used to power the Fears to Fathom series? (matched: 'unity')
     -> The Fears to Fathom series utilizes Unity as its primary game engine. Developed and published by Australian
[✗] When did the first season of Fears to Fathom officially debut?
     

## 13. Results

`base_*` came from the pristine model in section 5, `tuned_*` from the adapter
reloaded from disk in section 12 — two genuinely different models, scored by
the same function.

In [74]:
print(f"{'':<28}{'Accuracy':>10}")
print(f"{'-'*38}")
print(f"{'Base model':<28}{base_score:>9.1%}")
print(f"{'Fine-tuned (from disk)':<28}{tuned_score:>9.1%}")
print(f"{'-'*38}")
print(f"{'Gain':<28}{tuned_score - base_score:>+9.1%}")

print(f"""
How to read this:

- A clear positive gain on overlap means the paragraph's facts were absorbed
  into the weights and survived being asked about in different words.
- Roughly zero gain, with fluent but wrong answers, means training ran but the
  facts did not stick -- raise EPOCHS or LEARNING_RATE and rerun from section 9.
- Empty or garbled fine-tuned output means training damaged the model's ability
  to respond at all -- lower LEARNING_RATE (try 1e-4) and rerun from section 9.
""")

                              Accuracy
--------------------------------------
Base model                       0.0%
Fine-tuned (from disk)          25.0%
--------------------------------------
Gain                           +25.0%

How to read this:

- A clear positive gain on overlap means the paragraph's facts were absorbed
  into the weights and survived being asked about in different words.
- Roughly zero gain, with fluent but wrong answers, means training ran but the
  facts did not stick -- raise EPOCHS or LEARNING_RATE and rerun from section 9.
- Empty or garbled fine-tuned output means training damaged the model's ability
  to respond at all -- lower LEARNING_RATE (try 1e-4) and rerun from section 9.

